# Llama 3.1 8B QLoRA Fine-Tuning for ERP Domain Adaptation
This notebook fine-tunes Llama 3.1 8B using PEFT/QLoRA for broad ERP context awareness, evaluates it on a held-out ERP set, saves checkpoints to Google Drive, supports resume after Colab disconnects, and saves the final model to Drive.

> Research design note: company-specific policies and role restrictions are handled by the RAG/policy retrieval layer, not learned directly by the fine-tuned model.

In [1]:
%%capture
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "xformers<0.0.27" "trl<0.9.0" peft accelerate bitsandbytes
!pip install rouge-score

### Google Drive and Persistent Training Configuration
Mount Drive first so checkpoints and final models survive Colab runtime disconnects. The notebook will use dataset files from Drive when available, otherwise it falls back to `/content/sample_data/`.

In [ ]:
from google.colab import drive
import os

drive.mount("/content/drive")

# Persistent project folders
PROJECT_DIR = "/content/drive/MyDrive/erp_research"
CHECKPOINT_DIR = os.path.join(PROJECT_DIR, "checkpoints")
FINAL_MODEL_DIR = os.path.join(PROJECT_DIR, "models", "erp_lora_final")
FINAL_GGUF_DIR = os.path.join(PROJECT_DIR, "models", "erp_gguf_final")
DATA_DIR = os.path.join(PROJECT_DIR, "data")

for folder in [CHECKPOINT_DIR, FINAL_MODEL_DIR, FINAL_GGUF_DIR, DATA_DIR]:
    os.makedirs(folder, exist_ok=True)

# Prefer persistent Drive copies. If they are not there yet, use Colab sample_data.
DATASET_FILE_DRIVE = os.path.join(DATA_DIR, "erp_whw_1000_balanced.jsonl")
EVAL_FILE_DRIVE = os.path.join(DATA_DIR, "erp_eval_realistic_50.jsonl")

DATASET_FILE = DATASET_FILE_DRIVE if os.path.exists(DATASET_FILE_DRIVE) else "/content/sample_data/erp_whw_1000_balanced.jsonl"
EVAL_FILE = EVAL_FILE_DRIVE if os.path.exists(EVAL_FILE_DRIVE) else "/content/sample_data/erp_eval_realistic_50.jsonl"

print("Checkpoint directory:", CHECKPOINT_DIR)
print("Training dataset:", DATASET_FILE)
print("Evaluation dataset:", EVAL_FILE)
print("Tip: copy both JSONL files into", DATA_DIR, "once, so you do not need to re-upload after a disconnect.")

In [2]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 1024  # Lower memory use; increase only if your examples are genuinely longer
dtype = None
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.9.12: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Unsloth: Will load unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit as a legacy tokenizer.


### 📌 Base Model Parameter Stats (Before LoRA)

In [3]:
# Record GPU memory and parameter count BEFORE LoRA adapters are added
import torch

total_params_base = sum(p.numel() for p in model.parameters())
trainable_params_base = sum(p.numel() for p in model.parameters() if p.requires_grad)

gpu_mem_before_lora = torch.cuda.memory_allocated() / 1e9  # GB

print(f"Total parameters     : {total_params_base:,}")
print(f"Trainable parameters : {trainable_params_base:,}")
print(f"GPU memory (before LoRA): {gpu_mem_before_lora:.2f} GB")


Total parameters     : 4,540,600,320
Trainable parameters : 525,602,816
GPU memory (before LoRA): 5.75 GB


In [4]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)

Unsloth 2026.9.12 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


### ⚡ LoRA Parameter Efficiency

In [5]:
# Show LoRA impact: how many parameters are actually being trained
total_params_lora = sum(p.numel() for p in model.parameters())
trainable_params_lora = sum(p.numel() for p in model.parameters() if p.requires_grad)
frozen_params_lora = total_params_lora - trainable_params_lora

gpu_mem_after_lora = torch.cuda.memory_allocated() / 1e9  # GB

trainable_pct = (trainable_params_lora / total_params_lora) * 100

print("=" * 55)
print("          LoRA PARAMETER EFFICIENCY REPORT")
print("=" * 55)
print(f"{'Total parameters':<30}: {total_params_lora:>15,}")
print(f"{'Frozen parameters':<30}: {frozen_params_lora:>15,}")
print(f"{'Trainable (LoRA) parameters':<30}: {trainable_params_lora:>15,}")
print(f"{'Trainable %':<30}: {trainable_pct:>14.4f}%")
print("-" * 55)
print(f"{'GPU Memory before LoRA':<30}: {gpu_mem_before_lora:>13.2f} GB")
print(f"{'GPU Memory after LoRA':<30}: {gpu_mem_after_lora:>13.2f} GB")
print(f"{'Memory overhead of LoRA':<30}: {gpu_mem_after_lora - gpu_mem_before_lora:>13.2f} GB")
print("=" * 55)
print(f"\n✅ LoRA trains only {trainable_pct:.4f}% of parameters — full fine-tuning would train 100%.")


          LoRA PARAMETER EFFICIENCY REPORT
Total parameters              :   4,582,543,360
Frozen parameters             :   4,540,600,320
Trainable (LoRA) parameters   :      41,943,040
Trainable %                   :         0.9153%
-------------------------------------------------------
GPU Memory before LoRA        :          5.75 GB
GPU Memory after LoRA         :          5.92 GB
Memory overhead of LoRA       :          0.17 GB

✅ LoRA trains only 0.9153% of parameters — full fine-tuning would train 100%.


### Load and Prepare the ERP Dataset
The dataset is formatted as normal chat messages: `system` stays the system instruction, `user` stays the user request, and `assistant` is the target response. This avoids mixing system text into the user message.

The fine-tuned model is trained for **general ERP context awareness**. Company-specific policies and role restrictions remain in the RAG/policy layer.

In [6]:
from datasets import load_dataset
from unsloth.chat_templates import get_chat_template
import os

tokenizer = get_chat_template(tokenizer, chat_template="llama-3.1")

if not os.path.exists(DATASET_FILE):
    raise FileNotFoundError(
        f"Training dataset not found: {DATASET_FILE}\n"
        f"Upload it to /content/sample_data/ or place a persistent copy in {DATA_DIR}."
    )

dataset = load_dataset("json", data_files=DATASET_FILE, split="train")

# 90% training / 10% internal validation
split_dataset = dataset.train_test_split(test_size=0.1, seed=3407)
train_dataset = split_dataset["train"]
eval_dataset = split_dataset["test"]

DEFAULT_ERP_SYSTEM_PROMPT = (
    "You are a domain-aware AI assistant for a traditional ERP system. "
    "You understand ERP terminology, transactions, business processes, and operational procedures. "
    "Answer only from the ERP task and context provided. "
    "Company-specific policies and user-role restrictions are supplied separately by the RAG policy layer."
)

def format_prompts(examples):
    conversations = []

    for system_msg, user_msg, assistant_msg in zip(
        examples["system"], examples["user"], examples["assistant"]
    ):
        # Preserve dataset roles correctly. Use the broad ERP prompt only when
        # the dataset system field is missing/blank.
        system_content = (
            system_msg.strip()
            if isinstance(system_msg, str) and system_msg.strip()
            else DEFAULT_ERP_SYSTEM_PROMPT
        )
        user_content = user_msg.strip() if isinstance(user_msg, str) else str(user_msg)
        assistant_content = assistant_msg.strip() if isinstance(assistant_msg, str) else str(assistant_msg)

        convo = [
            {"role": "system", "content": system_content},
            {"role": "user", "content": user_content},
            {"role": "assistant", "content": assistant_content},
        ]
        conversations.append(convo)

    formatted_texts = [
        tokenizer.apply_chat_template(
            convo,
            tokenize=False,
            add_generation_prompt=False,
        )
        for convo in conversations
    ]

    return {"text": formatted_texts}

train_dataset = train_dataset.map(format_prompts, batched=True)
eval_dataset = eval_dataset.map(format_prompts, batched=True)

print(f"Train size: {len(train_dataset)}")
print(f"Eval size : {len(eval_dataset)}")
print("Example formatted record:
")
print(train_dataset[0]["text"][:1500])

Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/900 [00:00<?, ? examples/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

Train size: 900
Eval size: 100


### 📊 Base Model Evaluation (Before Fine-Tuning)
Compute eval loss / perplexity on the **unmodified base model** so we have a baseline to compare against after training.

In [7]:
# ──────────────────────────────────────────────────────────
# BASE MODEL — Perplexity / Loss (BEFORE fine-tuning)
# This uses the 10% internal eval split of the training data
# to measure next-token prediction quality.
# ──────────────────────────────────────────────────────────
import math, torch

FastLanguageModel.for_inference(model)
model.eval()

def compute_eval_loss(mdl, tok, dataset, max_samples=100):
    """Compute average cross-entropy loss on a HuggingFace dataset."""
    total_loss, count = 0.0, 0
    n = min(max_samples, len(dataset))
    for sample in dataset.select(range(n)):
        text = sample["text"]
        inputs = tok(text, return_tensors="pt", truncation=True, max_length=max_seq_length).to("cuda")
        with torch.no_grad():
            out = mdl(**inputs, labels=inputs["input_ids"])
            if out.loss is not None and not torch.isnan(out.loss):
                total_loss += out.loss.item()
                count += 1
    return total_loss / count if count > 0 else float("nan")

print("Evaluating BASE model perplexity on internal eval split...")
base_eval_loss  = compute_eval_loss(model, tokenizer, eval_dataset, max_samples=100)
base_perplexity = math.exp(base_eval_loss)

print(f"\nBASE MODEL — Internal Eval Results")
print(f"  Eval Loss  : {base_eval_loss:.4f}")
print(f"  Perplexity : {base_perplexity:.2f}")

FastLanguageModel.for_training(model)
print("\nModel switched back to training mode.")


`use_return_dict` is deprecated! Use `return_dict` instead!


Evaluating BASE model perplexity on internal eval split...


/usr/local/lib/python3.13/dist-packages/unsloth/import_fixes.py:5429: UserWarning: 'has_cuda' is deprecated, please use 'torch.backends.cuda.is_built()'
  return original(name)
/usr/local/lib/python3.13/dist-packages/unsloth/import_fixes.py:5429: UserWarning: 'has_cudnn' is deprecated, please use 'torch.backends.cudnn.is_available()'
  return original(name)
/usr/local/lib/python3.13/dist-packages/unsloth/import_fixes.py:5429: UserWarning: 'has_mps' is deprecated, please use 'torch.backends.mps.is_built()'
  return original(name)
/usr/local/lib/python3.13/dist-packages/unsloth/import_fixes.py:5429: UserWarning: 'has_mkldnn' is deprecated, please use 'torch.backends.mkldnn.is_available()'
  return original(name)



BASE MODEL — Internal Eval Results
  Eval Loss  : 4.0865
  Perplexity : 59.53

Model switched back to training mode.


In [8]:
# ──────────────────────────────────────────────────────────
# ERP ACCURACY EVALUATION — Setup
#
# Evaluation dataset:  erp_eval_realistic_50.jsonl
#   - 40 in-scope ERP questions  (paraphrased in natural human language)
#   - 10 out-of-scope questions  (model should refuse these)
#
# Accuracy logic (NO formatting/WHW checks):
#   In-scope  → ROUGE-L(generated vs. reference) >= THRESHOLD  → correct
#   Out-scope → model refuses to answer the non-ERP request    → correct
#
# Why ROUGE-L?  It measures longest-common-subsequence overlap between
# the generated ERP answer and the expert reference answer, capturing
# whether the model got the right SAP transaction codes, steps, and
# domain terms — regardless of how it phrased them.
# ──────────────────────────────────────────────────────────
import json, torch
from rouge_score import rouge_scorer as rs_module

# ── Threshold ──────────────────────────────────────────────
# 0.35 is the primary accuracy bar.  A correct ERP answer produced by the
# fine-tuned model will reliably score >= 0.35.
# The base model answers in conversational prose and scores ~0.20–0.28;
# some will cross 0.35 (it knew the topic), most won't.
ROUGE_THRESHOLD = 0.35

# ── Refusal keywords ───────────────────────────────────────
# If ANY of these appear in the model output we count the out-of-scope
# question as correctly refused.
REFUSAL_KEYWORDS = [
    "out of scope", "outside the scope", "outside my scope",
    "not related to erp", "not an erp", "unrelated to erp",
    "cannot answer", "cannot assist", "cannot help",
    "i cannot", "i am unable", "i'm unable", "i'm not able",
    "action blocked", "non-erp",
    "not authorized", "not permitted", "refuse",
    "this is not", "that is not", "does not relate",
]

def check_refusal(text: str) -> bool:
    lower = text.lower()
    return any(kw in lower for kw in REFUSAL_KEYWORDS)

# ── Load evaluation dataset ────────────────────────────────
if not os.path.exists(EVAL_FILE):
    raise FileNotFoundError(
        f"Evaluation dataset not found: {EVAL_FILE}\n"
        f"Upload it to /content/sample_data/ or place a persistent copy in {DATA_DIR}."
    )
eval_records = []
with open(EVAL_FILE, encoding="utf-8") as fh:
    for line in fh:
        if line.strip():
            eval_records.append(json.loads(line))

in_scope_count  = sum(1 for r in eval_records if "ACTION BLOCKED" not in r["assistant"])
out_scope_count = sum(1 for r in eval_records if "ACTION BLOCKED" in r["assistant"])

print(f"Loaded {len(eval_records)} evaluation records from {EVAL_FILE}")
print(f"  In-scope ERP questions : {in_scope_count}")
print(f"  Out-of-scope questions : {out_scope_count}")
print(f"  ROUGE-L threshold      : {ROUGE_THRESHOLD}")


# ── Core evaluation function ───────────────────────────────
def run_erp_eval(mdl, tok, records, max_new_tokens=256, label="", threshold=ROUGE_THRESHOLD):
    """
    Evaluate model ERP accuracy on a list of {system, user, assistant} records.

    Returns
    -------
    dict with:
      accuracy        – overall correct / total * 100
      in_scope_acc    – accuracy on the 40 ERP questions
      out_scope_acc   – refusal rate on the 10 non-ERP questions
      avg_rouge_l     – mean ROUGE-L across all 50 samples
      results         – per-sample detail list
    """
    FastLanguageModel.for_inference(mdl)
    mdl.eval()
    scorer = rs_module.RougeScorer(["rougeL"], use_stemmer=True)
    results = []

    for i, sample in enumerate(records):
        system_txt = sample["system"]
        user_txt   = sample["user"]
        reference  = sample["assistant"].strip()
        is_out     = "ACTION BLOCKED" in reference

        # Build chat prompt
        messages = [
            {"role": "system", "content": system_txt},
            {"role": "user",   "content": user_txt},
        ]
        inputs = tok.apply_chat_template(
            messages, tokenize=True,
            add_generation_prompt=True, return_tensors="pt"
        ).to("cuda")

        with torch.no_grad():
            out_ids = mdl.generate(
                input_ids=inputs,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                temperature=1.0,
                use_cache=True,
            )
        generated = tok.decode(out_ids[0][inputs.shape[-1]:], skip_special_tokens=True).strip()

        # Measure content overlap
        rouge_l = scorer.score(reference, generated)["rougeL"].fmeasure

        # Correctness decision
        if is_out:
            correct = check_refusal(generated)
        else:
            correct = (rouge_l >= threshold)

        results.append({
            "user":       user_txt,
            "reference":  reference,
            "generated":  generated,
            "rouge_l":    rouge_l,
            "correct":    correct,
            "out_of_scope": is_out,
        })

        if (i + 1) % 10 == 0 or (i + 1) == len(records):
            n_correct = sum(r["correct"] for r in results)
            print(f"  [{label}] {i+1}/{len(records)} evaluated  "
                  f"(running accuracy: {n_correct/(i+1)*100:.1f}%)")

    in_s = [r for r in results if not r["out_of_scope"]]
    out_s = [r for r in results if r["out_of_scope"]]

    return {
        "accuracy":      sum(r["correct"] for r in results) / len(results) * 100,
        "in_scope_acc":  sum(r["correct"] for r in in_s) / len(in_s) * 100 if in_s else 0.0,
        "out_scope_acc": sum(r["correct"] for r in out_s) / len(out_s) * 100 if out_s else 0.0,
        "avg_rouge_l":   sum(r["rouge_l"] for r in results) / len(results),
        "results":       results,
        "n":             len(results),
        "threshold":     threshold,
    }

print("\nEvaluation helper ready — run the next cell to evaluate the BASE model.")


Loaded 50 evaluation records from /content/sample_data/erp_eval_realistic_50.jsonl
  In-scope ERP questions : 40
  Out-of-scope questions : 10
  ROUGE-L threshold      : 0.35

Evaluation helper ready — run the next cell to evaluate the BASE model.


In [9]:
# ──────────────────────────────────────────────────────────
# BASE MODEL — ERP Accuracy (BEFORE fine-tuning)
# ──────────────────────────────────────────────────────────
print(f"Evaluating BASE model on {len(eval_records)} held-out ERP questions...")
print("(Each sample requires one forward + generate pass — ~3–5 min total)\n")

base_results = run_erp_eval(model, tokenizer, eval_records, label="BASE")

base_accuracy  = base_results["accuracy"]
base_rouge_l   = base_results["avg_rouge_l"]

print(f"\n{'='*50}")
print(f"BASE MODEL ERP Accuracy  ({base_results['n']} samples)")
print(f"{'='*50}")
print(f"  Overall Accuracy       : {base_accuracy:.2f}%")
print(f"  In-Scope ERP Accuracy  : {base_results['in_scope_acc']:.2f}%  (ROUGE-L >= {base_results['threshold']})")
print(f"  Out-of-Scope Refusal   : {base_results['out_scope_acc']:.2f}%")
print(f"  Average ROUGE-L        : {base_rouge_l:.4f}")
print(f"{'='*50}")

# Switch back to training mode so LoRA training can proceed
FastLanguageModel.for_training(model)
print("\nModel switched back to training mode — ready to fine-tune.")


The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Both `max_new_tokens` (=256) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Evaluating BASE model on 50 held-out ERP questions...
(Each sample requires one forward + generate pass — ~3–5 min total)



Both `max_new_tokens` (=256) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_

  [BASE] 10/50 evaluated  (running accuracy: 0.0%)


Both `max_new_tokens` (=256) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_

...

  [BASE] 30/50 evaluated  (running accuracy: 0.0%)


Both `max_new_tokens` (=256) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_

  [BASE] 40/50 evaluated  (running accuracy: 0.0%)


Both `max_new_tokens` (=256) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_

  [BASE] 50/50 evaluated  (running accuracy: 8.0%)

BASE MODEL ERP Accuracy  (50 samples)
  Overall Accuracy       : 8.00%
  In-Scope ERP Accuracy  : 0.00%  (ROUGE-L >= 0.35)
  Out-of-Scope Refusal   : 40.00%
  Average ROUGE-L        : 0.2226

Model switched back to training mode — ready to fine-tune.


### Train with Evaluation

In [10]:
from trl import SFTTrainer
from transformers import TrainingArguments
from transformers.trainer_utils import get_last_checkpoint
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    dataset_num_proc=2,
    # Keep False so internal eval examples remain independent and easy to compare.
    # Packing can be tested later as a separate speed optimization.
    packing=False,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=10,
        num_train_epochs=1,
        learning_rate=2e-4,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=5,
        eval_strategy="steps",
        eval_steps=25,

        # Save frequently to persistent Google Drive storage.
        save_strategy="steps",
        save_steps=25,
        save_total_limit=3,

        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=3407,
        output_dir=CHECKPOINT_DIR,
        report_to="none",
    ),
)

# Automatically continue from the newest checkpoint after a Colab disconnect.
latest_checkpoint = get_last_checkpoint(CHECKPOINT_DIR)

if latest_checkpoint:
    print("Resuming training from:", latest_checkpoint)
    trainer_stats = trainer.train(resume_from_checkpoint=latest_checkpoint)
else:
    print("No checkpoint found. Starting a new training run.")
    trainer_stats = trainer.train()

print("Training finished.")

Unsloth: transformers renamed `push_to_hub_token` to `hub_token`. Forwarding your value to `hub_token` - update your code when convenient. If you also passed `hub_token` as None, that is its default here and cannot be distinguished from leaving it unset, so `push_to_hub_token` was used; drop `push_to_hub_token` to keep it.
Unsloth: We found double BOS tokens - we shall remove one automatically.


Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/900 [00:00<?, ? examples/s]

Unsloth: We found double BOS tokens - we shall remove one automatically.


Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/100 [00:00<?, ? examples/s]

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 900 | Num Epochs = 1 | Total steps = 113
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 41,943,040 of 8,072,204,288 (0.52% trained)


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss,Validation Loss
10,2.877656,2.095329
20,1.152524,0.981878
30,0.735522,0.700047
40,0.554950,0.451933
50,0.284348,0.263199
60,0.207310,0.170084
70,0.123330,0.130574
80,0.111088,0.104429
90,0.081403,0.097538
100,0.087934,0.079787


Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-113/tokenizer_config.json.


### Evaluation Methods
Let's run an evaluation manually to see the loss, and then test the model with a sample prompt.

In [11]:
# ──────────────────────────────────────────────────────────
# POST-TRAINING: Loss / Perplexity Comparison
# ──────────────────────────────────────────────────────────
import math

eval_logs = [log for log in trainer.state.log_history if "eval_loss" in log]
if eval_logs:
    last_eval = eval_logs[-1]
    ft_eval_loss  = last_eval["eval_loss"]
    ft_perplexity = math.exp(ft_eval_loss)

    print("=" * 60)
    print("     TRAINING RESULTS: BASE vs FINE-TUNED MODEL")
    print("=" * 60)
    print(f"{'Metric':<22} {'Base Model':>12} {'Fine-Tuned':>12} {'Delta':>10}")
    print("-" * 60)
    print(f"{'Eval Loss':<22} {base_eval_loss:>12.4f} {ft_eval_loss:>12.4f} {(base_eval_loss - ft_eval_loss):>+10.4f}")
    print(f"{'Perplexity':<22} {base_perplexity:>12.2f} {ft_perplexity:>12.2f} {(base_perplexity - ft_perplexity):>+10.2f}")
    print("=" * 60)
    ppl_drop = (base_perplexity - ft_perplexity) / base_perplexity * 100
    print(f"\nPerplexity reduced by {ppl_drop:.1f}% — model learned the ERP domain.")
else:
    print("No evaluation logs found in trainer.state.log_history.")
    ft_eval_loss  = float("nan")
    ft_perplexity = float("nan")


     TRAINING RESULTS: BASE vs FINE-TUNED MODEL
Metric                   Base Model   Fine-Tuned      Delta
------------------------------------------------------------
Eval Loss                    4.0865       0.0742    +4.0122
Perplexity                    59.53         1.08     +58.45

Perplexity reduced by 98.2% — model learned the ERP domain.


### Fine-Tuned Model Accuracy (ERP Domain) — the Headline Comparison

In [12]:
# ──────────────────────────────────────────────────────────
# FINE-TUNED MODEL — ERP Accuracy (AFTER fine-tuning)
# Identical evaluation setup to the base model run above.
# ──────────────────────────────────────────────────────────
print(f"Evaluating FINE-TUNED model on {len(eval_records)} held-out ERP questions...")
print("(~3–5 min total)\n")

ft_results = run_erp_eval(model, tokenizer, eval_records, label="FT")

ft_accuracy = ft_results["accuracy"]
ft_rouge_l  = ft_results["avg_rouge_l"]

print(f"\n{'='*50}")
print(f"FINE-TUNED MODEL ERP Accuracy  ({ft_results['n']} samples)")
print(f"{'='*50}")
print(f"  Overall Accuracy       : {ft_accuracy:.2f}%")
print(f"  In-Scope ERP Accuracy  : {ft_results['in_scope_acc']:.2f}%  (ROUGE-L >= {ft_results['threshold']})")
print(f"  Out-of-Scope Refusal   : {ft_results['out_scope_acc']:.2f}%")
print(f"  Average ROUGE-L        : {ft_rouge_l:.4f}")
print(f"{'='*50}")

# ── Head-to-head comparison ────────────────────────────────
acc_delta   = ft_accuracy - base_accuracy
rouge_delta = ft_rouge_l - base_rouge_l

print("\n" + "=" * 64)
print("   ERP DOMAIN ACCURACY: BASE MODEL vs FINE-TUNED MODEL")
print("=" * 64)
print(f"{'Metric':<28} {'Base':>9} {'Fine-Tuned':>11} {'Delta':>10}")
print("-" * 64)
print(f"{'Overall Accuracy (%)':<28} {base_accuracy:>9.2f} {ft_accuracy:>11.2f} {acc_delta:>+10.2f}")
print(f"{'In-Scope ERP Accuracy (%)':<28} {base_results['in_scope_acc']:>9.2f} {ft_results['in_scope_acc']:>11.2f} {(ft_results['in_scope_acc'] - base_results['in_scope_acc']):>+10.2f}")
print(f"{'Out-of-Scope Refusal (%)':<28} {base_results['out_scope_acc']:>9.2f} {ft_results['out_scope_acc']:>11.2f} {(ft_results['out_scope_acc'] - base_results['out_scope_acc']):>+10.2f}")
print(f"{'Average ROUGE-L':<28} {base_rouge_l:>9.4f} {ft_rouge_l:>11.4f} {rouge_delta:>+10.4f}")
print("=" * 64)
print(f"\nFine-tuning improved ERP accuracy by {acc_delta:+.2f} percentage points")
print(f"using only {trainable_pct:.4f}% of model parameters (LoRA).")


Both `max_new_tokens` (=256) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Evaluating FINE-TUNED model on 50 held-out ERP questions...
(~3–5 min total)



Both `max_new_tokens` (=256) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_

  [FT] 10/50 evaluated  (running accuracy: 50.0%)


Both `max_new_tokens` (=256) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_

...

  [FT] 30/50 evaluated  (running accuracy: 53.3%)


Both `max_new_tokens` (=256) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_

  [FT] 40/50 evaluated  (running accuracy: 45.0%)


Both `max_new_tokens` (=256) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_

  [FT] 50/50 evaluated  (running accuracy: 52.0%)

FINE-TUNED MODEL ERP Accuracy  (50 samples)
  Overall Accuracy       : 52.00%
  In-Scope ERP Accuracy  : 45.00%  (ROUGE-L >= 0.35)
  Out-of-Scope Refusal   : 80.00%
  Average ROUGE-L        : 0.3947

   ERP DOMAIN ACCURACY: BASE MODEL vs FINE-TUNED MODEL
Metric                            Base  Fine-Tuned      Delta
----------------------------------------------------------------
Overall Accuracy (%)              8.00       52.00     +44.00
In-Scope ERP Accuracy (%)         0.00       45.00     +45.00
Out-of-Scope Refusal (%)         40.00       80.00     +40.00
Average ROUGE-L                 0.2226      0.3947    +0.1721

Fine-tuning improved ERP accuracy by +44.00 percentage points
using only 0.9153% of model parameters (LoRA).


### 🏆 LoRA Fine-Tuning Impact Summary
Side-by-side qualitative comparison: base model vs fine-tuned model on an ERP prompt.

In [13]:
# ──────────────────────────────────────────────────────────
# FINAL IMPACT SUMMARY
# ──────────────────────────────────────────────────────────
import textwrap

W = 64

# Quick qualitative check: ask the fine-tuned model a sample ERP question
FastLanguageModel.for_inference(model)
sample_messages = [
    {"role": "system", "content": "You are a domain expert AI for the company's traditional ERP system. Answer ERP process questions clearly."},
    {"role": "user",   "content": "How do I post a goods receipt for purchase order 4512345678 in company code 1000?"},
]
inputs = tokenizer.apply_chat_template(
    sample_messages, tokenize=True,
    add_generation_prompt=True, return_tensors="pt"
).to("cuda")
with torch.no_grad():
    out_ids = model.generate(input_ids=inputs, max_new_tokens=200, do_sample=False, use_cache=True)
sample_response = tokenizer.decode(out_ids[0][inputs.shape[-1]:], skip_special_tokens=True).strip()

print("\n" + "=" * W)
print(" LoRA FINE-TUNING — RESEARCH IMPACT SUMMARY ".center(W, "="))
print("=" * W)

print("\n PARAMETER EFFICIENCY")
print("-" * W)
print(f"  {'Total parameters':<26}: {total_params_lora:>15,}")
print(f"  {'Trainable (LoRA)':<26}: {trainable_params_lora:>15,}  ({trainable_pct:.4f}%)")
print(f"  {'Frozen':<26}: {frozen_params_lora:>15,}")
print(f"  {'GPU memory overhead':<26}: {gpu_mem_after_lora - gpu_mem_before_lora:>14.2f} GB")

print("\n ERP CONTENT ACCURACY (held-out realistic eval set, 50 questions)")
print("-" * W)
print(f"  {'Metric':<28} {'Base':>8} {'Fine-Tuned':>11} {'Gain':>8}")
print(f"  {'-'*58}")
print(f"  {'Overall Accuracy (%)':<28} {base_accuracy:>8.2f} {ft_accuracy:>11.2f} {(ft_accuracy-base_accuracy):>+8.2f}")
print(f"  {'In-Scope ERP Accuracy (%)':<28} {base_results['in_scope_acc']:>8.2f} {ft_results['in_scope_acc']:>11.2f} {(ft_results['in_scope_acc']-base_results['in_scope_acc']):>+8.2f}")
print(f"  {'Out-of-Scope Refusal (%)':<28} {base_results['out_scope_acc']:>8.2f} {ft_results['out_scope_acc']:>11.2f} {(ft_results['out_scope_acc']-base_results['out_scope_acc']):>+8.2f}")
print(f"  {'Average ROUGE-L':<28} {base_rouge_l:>8.4f} {ft_rouge_l:>11.4f} {(ft_rouge_l-base_rouge_l):>+8.4f}")

print("\n TRAINING QUALITY (internal 10% validation split)")
print("-" * W)
print(f"  {'Metric':<28} {'Base':>8} {'Fine-Tuned':>11} {'Gain':>8}")
print(f"  {'-'*58}")
print(f"  {'Eval Loss':<28} {base_eval_loss:>8.4f} {ft_eval_loss:>11.4f} {(base_eval_loss-ft_eval_loss):>+8.4f}")
print(f"  {'Perplexity':<28} {base_perplexity:>8.2f} {ft_perplexity:>11.2f} {(base_perplexity-ft_perplexity):>+8.2f}")

print("\n SAMPLE: Fine-tuned model answer to ERP question")
print("-" * W)
print("  Q: How do I post a goods receipt for PO 4512345678 in company code 1000?")
print()
for line in textwrap.wrap(sample_response, width=W - 4):
    print(f"  {line}")

print("\n" + "=" * W)


Both `max_new_tokens` (=200) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



========== LoRA FINE-TUNING — RESEARCH IMPACT SUMMARY ==========

 PARAMETER EFFICIENCY
----------------------------------------------------------------
  Total parameters          :   4,582,543,360
  Trainable (LoRA)          :      41,943,040  (0.9153%)
  Frozen                    :   4,540,600,320
  GPU memory overhead       :           0.17 GB

 ERP CONTENT ACCURACY (held-out realistic eval set, 50 questions)
----------------------------------------------------------------
  Metric                           Base  Fine-Tuned     Gain
  ----------------------------------------------------------
  Overall Accuracy (%)             8.00       52.00   +44.00
  In-Scope ERP Accuracy (%)        0.00       45.00   +45.00
  Out-of-Scope Refusal (%)        40.00       80.00   +40.00
  Average ROUGE-L                0.2226      0.3947  +0.1721

 TRAINING QUALITY (internal 10% validation split)
----------------------------------------------------------------
  Metric                           

In [14]:
# 2. Qualitative Evaluation (Inference)
FastLanguageModel.for_inference(model) # Enable native 2x faster inference

messages = [
    {"role": "system", "content": "You are a domain-aware AI assistant for a traditional ERP system. Answer ERP process questions clearly using the provided context."},
    {"role": "user", "content": "Classify the following into an ERP transaction.\nContext: Bought office supplies for $500."}
]

inputs = tokenizer.apply_chat_template(
    messages,
    tokenize = True,
    add_generation_prompt = True,
    return_tensors = "pt",
).to("cuda")

outputs = model.generate(input_ids = inputs, max_new_tokens = 64, use_cache = True)
print(tokenizer.batch_decode(outputs)[0])

Both `max_new_tokens` (=64) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 26 July 2024

You are an expert ERP Finance AI assistant. You process financial events accurately and do not hallucinate outside the provided context.<|eot_id|><|start_header_id|>user<|end_header_id|>

Classify the following into an ERP transaction.
Context: Bought office supplies for $500.<|eot_id|><|start_header_id|>assistant<|end_header_id|>

CLASSIFIED AS: Purchase Posting
WHAT: Record a procurement invoice for office supplies from vendor V100045 for $500.
WHAT NEXT: Match the invoice to PO 4510003029 or create a logistics invoice if purchasing is separate.<|eot_id|>


### Save the Model to Google Drive
The LoRA adapter, tokenizer, and optional GGUF export are saved in Drive so they remain available after the Colab runtime ends.

In [ ]:
# Save final LoRA adapter and tokenizer to persistent Google Drive
model.save_pretrained(FINAL_MODEL_DIR)
tokenizer.save_pretrained(FINAL_MODEL_DIR)
print("Saved LoRA adapter and tokenizer to:", FINAL_MODEL_DIR)

# Export a 4-bit GGUF for Ollama to persistent Google Drive.
# This can take additional time and Drive space. Comment this block out if you only need the LoRA adapter.
model.save_pretrained_gguf(
    FINAL_GGUF_DIR,
    tokenizer,
    quantization_method="q4_k_m",
)
print("Saved GGUF export to:", FINAL_GGUF_DIR)

In [ ]:
from google.colab import files
import os

# Optional: download one GGUF file to your computer.
# The model is already safe in Google Drive, so downloading is not required.
if os.path.exists(FINAL_GGUF_DIR):
    gguf_files = [f for f in os.listdir(FINAL_GGUF_DIR) if f.endswith(".gguf")]
    if gguf_files:
        gguf_file = os.path.join(FINAL_GGUF_DIR, gguf_files[0])
        print("GGUF stored safely in Drive:", gguf_file)
        print("If you want a local copy, uncomment the next line:")
        print(f'# files.download("{gguf_file}")')
    else:
        print("No GGUF file found in:", FINAL_GGUF_DIR)
else:
    print("GGUF directory does not exist:", FINAL_GGUF_DIR)